# DGCA domestic airline traffic: download and adjoin pipeline
Project 12, Airlines after COVID. Covers rubric sections 1 (collection) and 2 (storage).

**Run it locally, not on Colab.** Selenium works on Colab only with extra setup, and Indian government sites often time out or block cloud IPs. The notebook skips Selenium on Colab automatically and falls back to the file list from GitHub, so the download still works there.

**What the pipeline does**
1. Check robots.txt for every host (quote the output in your report).
2. Find the Excel links: Selenium on the DGCA portal, plus the URL list from the Vonter GitHub repo. The two lists are compared.
3. Download the Excel files from 2017 onward (airline-wise, domestic) into `data/raw/dgca_xlsx/`. Raw files are never edited.
4. Download the Vonter aggregated CSVs into `data/raw/github/` as the second source.
5. Profile every raw file, parse and stack them into one table in `data/interim/`, then cross-check against the GitHub CSV.

The Excel files are public S3 links, so they download directly with `requests`. Selenium is only used to discover links, not to click-download.

In [1]:
# %pip install -q selenium requests pandas openpyxl pyarrow pdfplumber

In [2]:
import os, re, sys, time, hashlib, datetime as dt
from pathlib import Path
from urllib.parse import quote, unquote, urlsplit
import requests
import pandas as pd

def find_project_root():
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks':
        root = cwd.parent
    elif (cwd / 'notebooks').is_dir() or (cwd / 'data').is_dir():
        root = cwd
    elif (cwd.parent / 'notebooks').is_dir():
        root = cwd.parent
    else:
        raise FileNotFoundError("Could not locate DGCA root folder.")
    return root

PROJECT_ROOT = find_project_root()

# ---------- config ----------
TODAY = dt.date.today().isoformat()
START_YEAR = 2017            # keep files for this year onward
INCLUDE_CITYPAIR = False     # monthly city-pair files; not needed for the airline view
EXCLUDE = r"totalint|quikjet|bluedart"   # international totals and cargo-only carriers (not passenger)
RUN_SELENIUM = True

PORTAL_URL = "https://www.dgca.gov.in/digigov-portal/?baseLocale=en_US?page=259/4184/servicename"
S3_BASE = "https://public-prd-dgca.s3.ap-south-1.amazonaws.com"
GH_RAW = "https://raw.githubusercontent.com/Vonter/india-aviation-traffic/main"
HEADERS = {"User-Agent": "Mozilla/5.0 (university course project; contact: YOUR_EMAIL_HERE)"}

RAW, INTERIM, CLEAN = PROJECT_ROOT / 'data' / 'raw', PROJECT_ROOT / 'data' / 'interim', PROJECT_ROOT / 'data' / 'clean'
for p in (RAW / "dgca_xlsx", RAW / "github", RAW / "robots", RAW / 'portal_debug', INTERIM, CLEAN):
    p.mkdir(parents=True, exist_ok=True)
print("Project root: ", PROJECT_ROOT)
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    RUN_SELENIUM = False
    print("Colab detected: Selenium discovery skipped, using the GitHub URL list.")
print("collection date:", TODAY)

Project root:  C:\Users\subhranshu das\OneDrive\Desktop\DGCA_aviation_analysis
collection date: 2026-11-01


## 1. robots.txt and terms of use
Quote this output in the report. The terms of use are a page on the portal, so read them there and quote the relevant lines (footer of the DGCA site).

In [3]:
hosts = ["https://www.dgca.gov.in", S3_BASE, "https://raw.githubusercontent.com"]
for h in hosts:
    try:
        r = requests.get(h + "/robots.txt", headers=HEADERS, timeout=30)
        text = r.text if r.status_code == 200 else ""
        (RAW / "robots" / f"robots_{urlsplit(h).netloc}_{TODAY}.txt").write_text(text)
        print(f"=== {h}/robots.txt  HTTP {r.status_code}")
        print(text[:1500] if text else "(no robots.txt served)")
    except Exception as e:
        print(f"=== {h}: {e}")

=== https://www.dgca.gov.in/robots.txt  HTTP 404
(no robots.txt served)
=== https://public-prd-dgca.s3.ap-south-1.amazonaws.com/robots.txt  HTTP 403
(no robots.txt served)
=== https://raw.githubusercontent.com/robots.txt  HTTP 404
(no robots.txt served)


## 2a. Link discovery with Selenium (portal)
Opens the start page, collects the year-page links it contains (`monthlyStatistics/259/<id>/html`), opens each of those, and collects every Excel link on them (following any deeper `yearly/...` pages and table "Next" buttons). It prints each page and what it found as it goes. The page's HTML is saved to `data/raw/portal_debug/` for each visit, so if a page yields nothing you can see what Selenium saw.

In [4]:
from html import unescape
from selenium.webdriver.common.by import By

HEADLESS = True        # set False to watch the browser work
EXAMPLE_PATH = "monthlyStatistics/259/9644/html"     # a year page you gave me; always visited as a second seed

XL_RE = re.compile(r"jsp/dgca/[^\"'<>]+?\.xlsx?", re.I)           # Excel links as the portal writes them
CHILD_RE = re.compile(r"(?:monthlyStatistics|yearly)/\d+/\d+/html")  # links to sub-pages

def to_s3(u):
    u = u.strip()
    if u.startswith("http"):
        return u
    return u.replace("jsp/dgca", S3_BASE, 1) if u.startswith("jsp/dgca") else S3_BASE + "/" + u.lstrip("/")

def page_url(path):
    return f"https://www.dgca.gov.in/digigov-portal/?baseLocale=en_US?page={path}&mainundefined"

def crawl(drv, seeds, depth_max=3, max_pages=60, wait=25):
    """Visit the start page, then every sub-page it links to; collect Excel links on each. Prints as it goes."""
    (RAW / "portal_debug").mkdir(parents=True, exist_ok=True)
    excel, seen, queue = set(), set(), list(seeds)
    while queue and len(seen) < max_pages:
        label, url, depth = queue.pop(0)
        if label in seen:
            continue
        seen.add(label)
        print(f"\n[{len(seen)}] opening {label}", flush=True)
        try:
            drv.get(url)
        except Exception as e:
            print("    page failed to load:", repr(e)[:150], flush=True)
            continue
        for t in range(wait):                       # wait for the JavaScript to draw the links
            html = unescape(drv.page_source)
            if XL_RE.search(html) or set(CHILD_RE.findall(html)) - {label}:
                break
            if t == 0:
                print("    waiting for the page to fill in...", flush=True)
            time.sleep(1)
        time.sleep(2)

        found, kids = set(), set()
        for i in range(100):                        # also walk 'Next' buttons if the table is paginated
            html = unescape(drv.page_source)
            before = (len(found), len(kids))
            found |= {to_s3(m) for m in XL_RE.findall(html)}
            kids |= set(CHILD_RE.findall(html))
            nxt = drv.find_elements(By.CSS_SELECTOR, ".paginate_button.next:not(.disabled)")
            if not nxt or (i > 0 and before == (len(found), len(kids))):
                break
            drv.execute_script("arguments[0].click();", nxt[0]); time.sleep(1)
        kids.discard(label)
        (RAW / "portal_debug" / f"{len(seen):02d}_{label.replace('/', '_')}.html").write_text(html, errors="ignore")

        new = found - excel
        excel |= found
        print(f"    {len(found)} Excel links ({len(new)} new) | {len(kids)} sub-pages | total Excel so far: {len(excel)}", flush=True)
        for u in sorted(new)[:6]:
            print("      file:", unquote(u.rsplit("/", 1)[-1]))
        if len(new) > 6:
            print(f"      ... and {len(new) - 6} more")
        for k in sorted(kids):
            print("      sub-page:", k)
        if not found and not kids:
            anchors = drv.find_elements(By.CSS_SELECTOR, "a")
            print(f"    nothing matched. {len(anchors)} anchors on the page, first 10 (text | data-url | href):")
            for a in anchors[:10]:
                print("      ", (a.text or "")[:40].replace("\n", " "), "|", a.get_attribute("data-url"), "|", a.get_attribute("href"))
        if depth < depth_max:
            queue += [(k, page_url(k), depth + 1) for k in sorted(kids) if k not in seen]
    print(f"\ndone: {len(seen)} pages visited, {len(excel)} Excel URLs found", flush=True)
    return excel

def discover_with_selenium(start_url):
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    o = Options()
    for a in (["--headless=new"] if HEADLESS else []) + ["--no-sandbox", "--disable-dev-shm-usage", "--window-size=1400,1000"]:
        o.add_argument(a)
    drv = webdriver.Chrome(options=o)              # Selenium Manager fetches the driver automatically
    drv.set_page_load_timeout(60)
    try:
        return crawl(drv, [("start", start_url, 0), (EXAMPLE_PATH, page_url(EXAMPLE_PATH), 1)])
    finally:
        drv.quit()

selenium_urls = set()
if RUN_SELENIUM:
    try:
        selenium_urls = discover_with_selenium(PORTAL_URL)
    except Exception as e:
        print("Selenium discovery failed, continuing with the GitHub list only:", repr(e)[:300])
pd.Series(sorted(selenium_urls)).to_csv(RAW / f"selenium_urls_{TODAY}.csv", index=False, header=["url"])


[1] opening start
    0 Excel links (0 new) | 18 sub-pages | total Excel so far: 0
      sub-page: monthlyStatistics/259/9512/html
      sub-page: monthlyStatistics/259/9513/html
      sub-page: monthlyStatistics/259/9514/html
      sub-page: monthlyStatistics/259/9515/html
      sub-page: monthlyStatistics/259/9517/html
      sub-page: monthlyStatistics/259/9518/html
      sub-page: monthlyStatistics/259/9519/html
      sub-page: monthlyStatistics/259/9520/html
      sub-page: monthlyStatistics/259/9521/html
      sub-page: monthlyStatistics/259/9522/html
      sub-page: monthlyStatistics/259/9523/html
      sub-page: monthlyStatistics/259/9524/html
      sub-page: monthlyStatistics/259/9525/html
      sub-page: monthlyStatistics/259/9526/html
      sub-page: monthlyStatistics/259/9527/html
      sub-page: monthlyStatistics/259/9528/html
      sub-page: monthlyStatistics/259/9529/html
      sub-page: monthlyStatistics/259/9644/html

[2] opening monthlyStatistics/259/9644/html
    wai

## 2b. Second source: Vonter/india-aviation-traffic on GitHub
Licence: ODbL. It must be attributed to DGCA and the Ministry of Civil Aviation. The repo's `dgca/urls.txt` is its list of DGCA file URLs, and `aggregated/domestic/carrier.csv` is its parsed airline-wise table.

In [5]:
def get(url, **kw):
    r = requests.get(url, headers=HEADERS, timeout=60, **kw)
    r.raise_for_status()
    return r

# record the commit so the report can cite an exact version
try:
    sha = get("https://api.github.com/repos/Vonter/india-aviation-traffic/commits/main").json()["sha"]
except Exception:
    sha = "unknown"
print("repo commit:", sha)

gh_files = {
    "urls": ("dgca/urls.txt", f"vonter_dgca_urls_{TODAY}.txt"),
    "carrier": ("aggregated/domestic/carrier.csv", f"vonter_domestic_carrier_{TODAY}.csv"),
    "city": ("aggregated/domestic/city.csv", f"vonter_domestic_city_{TODAY}.csv"),
}
for key, (src, dst) in gh_files.items():
    if key == "city" and not INCLUDE_CITYPAIR:
        continue
    (RAW / "github" / dst).write_bytes(get(f"{GH_RAW}/{src}").content)
    print("saved", dst)

github_urls = {to_s3(u) if not u.startswith("http") else u
               for u in (RAW / "github" / gh_files["urls"][1]).read_text().splitlines() if u.strip()}
print(len(github_urls), "URLs in the GitHub list")

repo commit: e197170f131e43169620e4f8aa520350d4cbcba9
saved vonter_dgca_urls_2026-11-01.txt
saved vonter_domestic_carrier_2026-11-01.csv
533 URLs in the GitHub list


## 3. Build the download list
Keeps domestic, `START_YEAR` onward, airline-wise files (city-pair only if the flag is on). The year is the last two digits of airline-wise file names (`indigo19.xlsx` → 2019) and the 4-digit year in city-pair names. Each URL is tagged with where it was found.

In [6]:
def classify(url):
    name = unquote(urlsplit(url).path.rsplit("/", 1)[-1]).strip()
    stem = re.sub(r"\.xlsx?$", "", name, flags=re.I).strip()
    if "CITYPAIR" in name.upper():
        m = re.search(r"(20\d{2})", name); year = int(m.group(1)) if m else None
        return name, "citypair", year
    m = re.search(r"(\d{2})$", stem)
    return name, "airline", (2000 + int(m.group(1)) if m else None)

rows = []
for u in sorted(github_urls | selenium_urls):
    if "/domestic/" not in u:
        continue
    name, kind, year = classify(u)
    src = "both" if (u in github_urls and u in selenium_urls) else ("github" if u in github_urls else "selenium")
    rows.append(dict(url=u, original_name=name, kind=kind, year=year, found_by=src))
cand = pd.DataFrame(rows)

keep = cand[(cand.year >= START_YEAR) & ~cand.original_name.str.contains(EXCLUDE, case=False, regex=True)]
if not INCLUDE_CITYPAIR:
    keep = keep[keep.kind == "airline"]
keep = keep.reset_index(drop=True)

print("domestic URLs found:", len(cand), "| kept:", len(keep))
print(keep.found_by.value_counts().to_string())
print("files per year:\n", keep.year.value_counts().sort_index().to_string())
print("year not parsed:", cand[cand.year.isna()].original_name.tolist())

domestic URLs found: 349 | kept: 150
found_by
both    150
files per year:
 year
2017    17
2018    15
2019    15
2020    14
2021    16
2022    16
2023    15
2024    15
2025    14
2026    13
year not parsed: []


## 4. Download
Saved as `dgca_<name>_<collection date>.xlsx`, never overwritten. Every response is checked to be a real Excel file (an error page saved as .xlsx is the classic silent failure). All outcomes go to a manifest.

In [7]:
def slug(name):
    stem, ext = os.path.splitext(name)
    return (
        re.sub(r"[^a-z0-9]+", "_", stem.lower()).strip("_")
        + ext.lower()
    )


def is_excel(b):
    return (
        b[:4] == b"PK\x03\x04"
        or b[:8] == bytes.fromhex("d0cf11e0a1b11ae1")
    )


def download(url, name):
    s = slug(name)
    stem, ext = os.path.splitext(s)

    # Check whether this source file has already been downloaded.
    existing = sorted(
        (RAW / "dgca_xlsx").glob(f"dgca_{stem}_*{ext}")
    )

    if existing:
        cached_path = existing[-1]

        # Recover the original collection date from the filename.
        match = re.search(
            r"_(\d{4}-\d{2}-\d{2})$",
            cached_path.stem
        )

        original_date = (
            match.group(1) if match else None
        )

        content = cached_path.read_bytes()

        # Verify that the cached file is still a valid Excel file.
        if not is_excel(content):
            return {
                "status": "invalid_cached_file",
                "path": str(cached_path),
                "collected": original_date,
                "error": "Cached file is not a valid Excel-format file"
            }

        return {
            "status": "cached",
            "path": str(cached_path),
            "bytes": len(content),
            "sha256": hashlib.sha256(content).hexdigest(),
            "collected": original_date
        }

    # Download a new file if no cached version exists.
    safe = quote(unquote(url), safe=":/")

    for attempt in range(3):
        try:
            r = requests.get(
                safe,
                headers=HEADERS,
                timeout=(15, 90)
            )

            if r.status_code in (403, 404):
                return {
                    "status": f"http_{r.status_code}",
                    "path": "",
                    "collected": None,
                    "error": f"HTTP {r.status_code}"
                }

            r.raise_for_status()

            if not is_excel(r.content):
                raise ValueError(
                    "Response is not an Excel-format file"
                )

            p = (
                RAW / "dgca_xlsx" /
                f"dgca_{stem}_{TODAY}{ext}"
            )

            p.write_bytes(r.content)

            return {
                "status": "downloaded",
                "path": str(p),
                "bytes": len(r.content),
                "sha256": hashlib.sha256(r.content).hexdigest(),
                "collected": TODAY
            }

        except Exception as e:

            if attempt == 2:
                return {
                    "status": "error",
                    "path": "",
                    "collected": None,
                    "error": str(e)[:200]
                }

            time.sleep(2 * (attempt + 1))


# Ensure two source files do not produce the same filename.
assert keep.original_name.map(slug).is_unique, (
    "Two source files would collide on the same saved name"
)


# Download all selected DGCA files.
results = []

for i, r in keep.iterrows():

    out = download(r.url, r.original_name)

    results.append({
        **r.to_dict(),
        **out,
        "reused_on": (
            TODAY if out["status"] == "cached" else None
        )
    })

    if out["status"] == "downloaded":
        time.sleep(0.5)  # Be polite to the server.

    if (i + 1) % 25 == 0:
        print(f"{i + 1}/{len(keep)}")


# Save the download manifest.
manifest = pd.DataFrame(results)

manifest.to_csv(
    RAW / f"manifest_{TODAY}.csv",
    index=False
)


# Display collection summary.
print("\nDOWNLOAD SUMMARY")
print("----------------")

print(manifest.status.value_counts().to_string())

print("\nTotal source files:", len(manifest))
print(
    "Successfully available:",
    manifest.status.isin(["downloaded", "cached"]).sum()
)
print(
    "Failed:",
    (~manifest.status.isin(["downloaded", "cached"])).sum()
)

print("\nFAILED DOWNLOADS")

display(
    manifest[
        ~manifest.status.isin(["downloaded", "cached"])
    ][
        ["original_name", "url", "status", "error"]
    ]
)

25/150
50/150
75/150
100/150
125/150
150/150

DOWNLOAD SUMMARY
----------------
status
downloaded    142
http_403        8

Total source files: 150
Successfully available: 142
Failed: 8

FAILED DOWNLOADS


,original_name,url,status,error
31,airasia17.xlsx,https://public-prd-dgca.s3.ap-south-1.amazonaw...,http_403,HTTP 403
74,go air25.xlsx,https://public-prd-dgca.s3.ap-south-1.amazonaw...,http_403,HTTP 403
75,go air26.xlsx,https://public-prd-dgca.s3.ap-south-1.amazonaw...,http_403,HTTP 403
135,trujet24.xlsx,https://public-prd-dgca.s3.ap-south-1.amazonaw...,http_403,HTTP 403
136,trujet25.xlsx,https://public-prd-dgca.s3.ap-south-1.amazonaw...,http_403,HTTP 403
137,trujet26.xlsx,https://public-prd-dgca.s3.ap-south-1.amazonaw...,http_403,HTTP 403
146,vistara25.xlsx,https://public-prd-dgca.s3.ap-south-1.amazonaw...,http_403,HTTP 403
147,vistara26.xlsx,https://public-prd-dgca.s3.ap-south-1.amazonaw...,http_403,HTTP 403


## 5. Profile every raw file
Rubric: shape, dtypes and column meaning for each raw file before any change. This reads without modifying. The first sheet is used because these workbooks have one sheet each; the sheet count is logged so you can check.

In [8]:
ok=manifest[manifest.status.isin(["downloaded","cached"])].copy()
prof=[];sheet_profiles=[];column_profiles=[]
for _,r in ok.iterrows():
    try:
        with pd.ExcelFile(r.path) as xl:
            total_rows=0;total_non_null=0;max_cols=0
            for sheet in xl.sheet_names:
                df=xl.parse(sheet,header=None)
                rows,cols=df.shape
                non_null=int(df.notna().sum().sum())
                total_rows+=rows;total_non_null+=non_null;max_cols=max(max_cols,cols)
                sheet_profiles.append({"file":r.original_name,"sheet":sheet,"rows":rows,"columns":cols,"non_null_cells":non_null})
                for col in df.columns:
                    s=df[col].dropna()
                    numeric=pd.to_numeric(s.astype(str).str.replace(",","",regex=False),errors="coerce")
                    inferred="empty" if len(s)==0 else ("numeric-like" if numeric.notna().mean()>=0.8 else "text/mixed")
                    column_profiles.append({"file":r.original_name,"sheet":sheet,"column_position":col,"non_null":len(s),"inferred_content_type":inferred})
            prof.append({"file":r.original_name,"saved":r.path,"sheets":len(xl.sheet_names),"total_rows":total_rows,"max_columns":max_cols,"non_null_cells":total_non_null,"status":"profiled"})
    except Exception as e:
        prof.append({"file":r.original_name,"saved":r.path,"status":"error","error":str(e)[:200]})
profile=pd.DataFrame(prof)
sheet_profile=pd.DataFrame(sheet_profiles)
column_profile=pd.DataFrame(column_profiles)
profile.to_csv(INTERIM/f"raw_profile_{TODAY}.csv",index=False)
sheet_profile.to_csv(INTERIM/f"sheet_profile_{TODAY}.csv",index=False)
column_profile.to_csv(INTERIM/f"column_profile_{TODAY}.csv",index=False)
print("Files profiled:",int((profile.status=="profiled").sum()))
print("Files with profiling errors:",int((profile.status=="error").sum()))
print("Files with multiple sheets:",int((profile.loc[profile.status=="profiled","sheets"]>1).sum()))
display(profile.head())
display(sheet_profile.head())
display(column_profile.head())

C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2329841507.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2329841507.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2329841507.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2329841507.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2329841507.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2329841507.py:8: Pandas4Wa

Files profiled: 142
Files with profiling errors: 0
Files with multiple sheets: 83


C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2329841507.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2329841507.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2329841507.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2329841507.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2329841507.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2329841507.py:8: Pandas4Wa

,file,saved,sheets,total_rows,max_columns,non_null_cells,status
0,Air Asia21.xlsx,C:\Users\subhranshu das\OneDrive\Desktop\DGCA_...,2,494,17,584,profiled
1,Air Asia22.xlsx,C:\Users\subhranshu das\OneDrive\Desktop\DGCA_...,2,494,17,605,profiled
2,Air Asia23.xlsx,C:\Users\subhranshu das\OneDrive\Desktop\DGCA_...,2,494,17,580,profiled
3,Air India21.xlsx,C:\Users\subhranshu das\OneDrive\Desktop\DGCA_...,2,494,17,766,profiled
4,Air India22.xlsx,C:\Users\subhranshu das\OneDrive\Desktop\DGCA_...,2,494,17,704,profiled


,file,sheet,rows,columns,non_null_cells
0,Air Asia21.xlsx,Monthly_Template,494,17,584
1,Air Asia21.xlsx,Sheet1,0,0,0
2,Air Asia22.xlsx,Monthly_Template,494,17,605
3,Air Asia22.xlsx,Sheet1,0,0,0
4,Air Asia23.xlsx,Monthly_Template,494,17,580


,file,sheet,column_position,non_null,inferred_content_type
0,Air Asia21.xlsx,Monthly_Template,0,72,text/mixed
1,Air Asia21.xlsx,Monthly_Template,1,35,text/mixed
2,Air Asia21.xlsx,Monthly_Template,2,35,text/mixed
3,Air Asia21.xlsx,Monthly_Template,3,31,numeric-like
4,Air Asia21.xlsx,Monthly_Template,4,31,numeric-like


In [9]:
valid=profile[profile.status=="profiled"]
if not valid.empty:
    sample_file=valid.iloc[0]
    sample=pd.read_excel(sample_file.saved,header=None)
    print("File:",sample_file["file"])
    print("Shape:",sample.shape)
    print("Data types:\n",sample.dtypes.value_counts().to_string())
    display(sample.head(25))
else:
    print("No successfully profiled Excel files available for inspection.")

File: Air Asia21.xlsx
Shape: (494, 17)
Data types:
 object    17


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,2021,NaN,Monthly Traffic And Operating Statistics (Prov...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AIR ASIA,NaN,NaN
1,MONTH,AIRCRAFT FLOWN,NaN,NaN,PASSENGERS CARRIED\n(IN NUMBER),PASSENGER KMS.PERFORMED\n(IN THOUSAND),AVAILABLE SEAT KILOMETRE\n(IN THOUSAND),PAX.LOAD FACTOR#\n(IN %),CARGO CARRIED,NaN,NaN,TONNE KILOMETRE PERFORMED,NaN,NaN,NaN,AVAILABLE TONNE KILOMETRE\n(IN THOUSAND),WEIGHT LOAD FACTOR##\n(IN %)
2,NaN,DEPARTURES\n(IN NUMBER),HOURS\n(IN NUMBER),KILOMETRE\n(IN THOUSAND),NaN,NaN,NaN,NaN,FREIGHT\n(IN TONNE),MAIL\n(IN TONNE),TOTAL\n(IN TONNE),PASSENGER\n(IN THOUSAND),FREIGHT\n(IN THOUSAND),MAIL\n(IN THOUSAND),TOTAL\n(IN THOUSAND),NaN,NaN
3,JAN,4495,8590.4,4621.139,531578,558871.307,836264.022,66.829529,3175.7262,230.1885,3405.9147,37494.472877,4067.437374,269.242257,41831.152508,92422.78,45.260652
4,FEB,4340,8239.033333,4417,520603,542276.763,798746.808,67.890946,3410.6377,211.8922,3622.5299,39555.584103,4264.010976,251.988624,44071.583703,88341.12,49.887961
5,MAR,4727,9023.3,4855.674,541512,572123.104,878405.634,65.131994,3662.2624,211.7535,3874.0159,38524.441947,4577.131657,280.847992,43382.421596,97113.48,44.671884
6,APR,3195,6341.383333,3481.586,354878,403399.335,630122.874,64.019154,2487.1216,169.238,2656.3596,27963.79866,3332.789591,208.044115,31504.632366,69631.72,45.244656
7,MAY,809,1562.15,867.183,63532,70007.636,157745.28,44.380178,1344.1482,57.5075,1401.6557,4039.994933,1744.005877,114.999778,5899.000588,17343.66,34.012432
8,JUNE,1081,1998.433333,1082.085,106632,107461.227,197372.13,54.445999,999.862,97.4495,1097.3115,5674.441101,2390.072116,191.921503,8256.43472,21641.7,38.150583
9,JULY,1721,3141.533333,1700.45,164608,165951.701,309698.724,53.584884,1453.455,135.3095,1588.7645,8240.750566,3073.852855,270.48802,11585.091441,34009.0,34.064781


## 6. Parse and adjoin into one table
Layout assumption, taken from how the Vonter repo parses the same files: one sheet per airline-year, split into sections (Scheduled/Non-scheduled × Domestic/International) with one row per month, month name in column 0 and 16 metrics after it.

**This parser is untested against the real files** (the sandbox I wrote it in cannot reach DGCA's S3 bucket). Step 7 compares its output to the Vonter CSV, which is the real test. If the match rate is low, fix the parser using the sample printed above.

In [10]:
multi=sheet_profile[sheet_profile.file.isin(profile.loc[profile.sheets>1,"file"])]
if not multi.empty:
    name=multi.file.iloc[0]
    path=profile.loc[profile.file==name,"saved"].iloc[0]
    with pd.ExcelFile(path) as xl:
        print("File:",name,"| Sheets:",xl.sheet_names)
        for sh in xl.sheet_names[:5]:
            d=xl.parse(sh,header=None)
            print("\nSHEET:",sh,"| SHAPE:",d.shape)
            display(d.head(15))
else:
    print("No multi-sheet workbooks found.")

File: Air Asia21.xlsx | Sheets: ['Monthly_Template', 'Sheet1']

SHEET: Monthly_Template | SHAPE: (494, 17)


C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\4179808683.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  d=xl.parse(sh,header=None)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,2021,NaN,Monthly Traffic And Operating Statistics (Prov...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AIR ASIA,NaN,NaN
1,MONTH,AIRCRAFT FLOWN,NaN,NaN,PASSENGERS CARRIED\n(IN NUMBER),PASSENGER KMS.PERFORMED\n(IN THOUSAND),AVAILABLE SEAT KILOMETRE\n(IN THOUSAND),PAX.LOAD FACTOR#\n(IN %),CARGO CARRIED,NaN,NaN,TONNE KILOMETRE PERFORMED,NaN,NaN,NaN,AVAILABLE TONNE KILOMETRE\n(IN THOUSAND),WEIGHT LOAD FACTOR##\n(IN %)
2,NaN,DEPARTURES\n(IN NUMBER),HOURS\n(IN NUMBER),KILOMETRE\n(IN THOUSAND),NaN,NaN,NaN,NaN,FREIGHT\n(IN TONNE),MAIL\n(IN TONNE),TOTAL\n(IN TONNE),PASSENGER\n(IN THOUSAND),FREIGHT\n(IN THOUSAND),MAIL\n(IN THOUSAND),TOTAL\n(IN THOUSAND),NaN,NaN
3,JAN,4495,8590.4,4621.139,531578,558871.307,836264.022,66.829529,3175.7262,230.1885,3405.9147,37494.472877,4067.437374,269.242257,41831.152508,92422.78,45.260652
4,FEB,4340,8239.033333,4417,520603,542276.763,798746.808,67.890946,3410.6377,211.8922,3622.5299,39555.584103,4264.010976,251.988624,44071.583703,88341.12,49.887961
5,MAR,4727,9023.3,4855.674,541512,572123.104,878405.634,65.131994,3662.2624,211.7535,3874.0159,38524.441947,4577.131657,280.847992,43382.421596,97113.48,44.671884
6,APR,3195,6341.383333,3481.586,354878,403399.335,630122.874,64.019154,2487.1216,169.238,2656.3596,27963.79866,3332.789591,208.044115,31504.632366,69631.72,45.244656
7,MAY,809,1562.15,867.183,63532,70007.636,157745.28,44.380178,1344.1482,57.5075,1401.6557,4039.994933,1744.005877,114.999778,5899.000588,17343.66,34.012432
8,JUNE,1081,1998.433333,1082.085,106632,107461.227,197372.13,54.445999,999.862,97.4495,1097.3115,5674.441101,2390.072116,191.921503,8256.43472,21641.7,38.150583
9,JULY,1721,3141.533333,1700.45,164608,165951.701,309698.724,53.584884,1453.455,135.3095,1588.7645,8240.750566,3073.852855,270.48802,11585.091441,34009.0,34.064781



SHEET: Sheet1 | SHAPE: (0, 0)


C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\4179808683.py:8: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  d=xl.parse(sh,header=None)


""


In [11]:
FULL_MONTHS=["JANUARY","FEBRUARY","MARCH","APRIL","MAY","JUNE","JULY","AUGUST","SEPTEMBER","OCTOBER","NOVEMBER","DECEMBER"]
def month_num(x):
    if not isinstance(x,str): return None
    k=re.sub(r"[^A-Za-z]","",x).upper()
    if k=="FEBURUARY": return 2
    if k=="SEPT": return 9
    for i,m in enumerate(FULL_MONTHS,1):
        if k==m or (len(k)>=3 and m.startswith(k)): return i
    return None

SECTIONS=[
    (r"NON[\s-]*SCHED.*INTER","NonScheduledInternational"),
    (r"(?<!NON[\s-])SCHED.*INTER","ScheduledInternational"),
    (r"NON[\s-]*SCHED.*DOM","NonScheduledDomestic"),
    (r"(?<!NON[\s-])SCHED.*DOM","ScheduledDomestic")
]
METRICS=["Aircraft Number","Aircraft Hours","Aircraft Kilometres","Passenger Number","Passenger Kilometers","Seat Kilometers","Passenger Load Factor","Freight","Mail","Total Cargo","Passenger Tonne Kilometer","Mail Tonne Kilometer","Freight Tonne Kilometer","Total Tonne Kilometer","Available Tonne Kilometer","Weight Load Factor"]
AIRLINE_MAP={k.lower():v for k,v in {
    "Air Asia":"AirAsia India","airasia":"AirAsia India","Air India":"Air India","airindia":"Air India",
    "air india express":"Air India Express","airindiaexpress":"Air India Express","air taxi":"Air Taxi",
    "Flybig":"Flybig","Spicejet":"SpiceJet","TruJet":"TruJet","TrueJet":"TruJet","air deccan":"Air Deccan","airdeccan":"Air Deccan",
    "deccanair":"Air Deccan","air heritage":"Air Heritage","airheritage":"Air Heritage","aircarnival":"Air Carnival",
    "aircosta":"Air Costa","airodisha":"Air Odisha","airpegasus":"Air Pegasus","aix connect":"AIX Connect",
    "akasa air":"Akasa Air","alliance":"Alliance Air","alliance air":"Alliance Air","fly":"Fly91",
    "go air":"Go First","goair":"Go First","india one air":"India One Air","indigo":"IndiGo",
    "jetairways":"Jet Airways","jetlite":"JetLite","pawan hans":"Pawan Hans","pawanhans":"Pawan Hans",
    "star air":"Star Air","starair":"Star Air","totaldom":"Total Domestic","vistara":"Vistara","zoomair":"Zoom Air"
}.items()}

def airline_from_name(name):
    base=re.sub(r"\d+","",re.sub(r"\.xlsx?$","",name,flags=re.I)).strip().lower()
    return AIRLINE_MAP.get(base),base

def parse_workbook(path,original_name,year):
    airline,base=airline_from_name(original_name)
    records=[];dash_count=0
    with pd.ExcelFile(path) as xl:
        for sheet in xl.sheet_names:
            df=xl.parse(sheet,header=None)
            current_section=None
            for _,row in df.iterrows():
                cells=[str(v) for v in row.tolist() if pd.notna(v)]
                row_text=" ".join(cells)
                m=month_num(row.iloc[0]) if len(row)>0 else None
                if m is None:
                    for pattern,label in SECTIONS:
                        if re.search(pattern,row_text,re.I):
                            current_section=label
                            break
                    continue
                if current_section is None:
                    continue
                vals=row.iloc[1:17].tolist()
                vals+=([None]*(16-len(vals)))
                dash_count+=sum(isinstance(v,str) and v.strip()=="-" for v in vals)
                records.append([current_section,airline or base,year,m,original_name,sheet]+vals)
    cols=["Type","Airline","Year","Month","Source File","Source Sheet"]+METRICS
    return pd.DataFrame(records,columns=cols),dash_count,airline is None

frames=[];parse_log=[]
for _,r in ok[ok.kind=="airline"].dropna(subset=["year"]).iterrows():
    try:
        t,dashes,unmapped=parse_workbook(r.path,r.original_name,int(r.year))
        if not t.empty:
            frames.append(t)
        parse_log.append({"file":r.original_name,"rows_parsed":len(t),"dash_cells":dashes,"airline_unmapped":unmapped,"status":"parsed"})
    except Exception as e:
        parse_log.append({"file":r.original_name,"rows_parsed":0,"dash_cells":0,"airline_unmapped":False,"status":"error","error":str(e)[:200]})

parse_log=pd.DataFrame(parse_log)
if not frames:
    raise ValueError("No rows were parsed. Inspect the raw workbook layout and section labels before continuing.")
dgca=pd.concat(frames,ignore_index=True)
for c in METRICS:
    s=dgca[c].astype("string").str.replace(",","",regex=False).str.strip().replace("-",pd.NA)
    dgca[c]=pd.to_numeric(s,errors="coerce")
print("Parsed rows:",len(dgca))
print("Files with parsing errors:",int((parse_log.status=="error").sum()))
print("Files with zero parsed rows:",int((parse_log.rows_parsed==0).sum()))
print("Unmapped airline names:",parse_log.loc[parse_log.airline_unmapped,"file"].tolist())
display(parse_log[parse_log.status!="parsed"])
display(dgca.head())
dgca.head()

C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2716402985.py:39: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2716402985.py:39: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2716402985.py:39: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2716402985.py:39: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2716402985.py:39: Pandas4Warning: ExcelFile.parse is deprecated. Use pd.read_excel instead.
  df=xl.parse(sheet,header=None)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\2716402985.py:39: Pan

Parsed rows: 5844
Files with parsing errors: 0
Files with zero parsed rows: 0
Unmapped airline names: []


,file,rows_parsed,dash_cells,airline_unmapped,status


,Type,Airline,Year,Month,Source File,Source Sheet,Aircraft Number,Aircraft Hours,Aircraft Kilometres,Passenger Number,...,Passenger Load Factor,Freight,Mail,Total Cargo,Passenger Tonne Kilometer,Mail Tonne Kilometer,Freight Tonne Kilometer,Total Tonne Kilometer,Available Tonne Kilometer,Weight Load Factor
0,ScheduledDomestic,AirAsia India,2021,1,Air Asia21.xlsx,Monthly_Template,4495.0,8590.4,4621.139,531578.0,...,66.829529,3175.7262,230.1885,3405.9147,37494.472877,4067.437374,269.242257,41831.152508,92422.78,45.260652
1,ScheduledDomestic,AirAsia India,2021,2,Air Asia21.xlsx,Monthly_Template,4340.0,8239.033333,4417.0,520603.0,...,67.890946,3410.6377,211.8922,3622.5299,39555.584103,4264.010976,251.988624,44071.583703,88341.12,49.887961
2,ScheduledDomestic,AirAsia India,2021,3,Air Asia21.xlsx,Monthly_Template,4727.0,9023.3,4855.674,541512.0,...,65.131994,3662.2624,211.7535,3874.0159,38524.441947,4577.131657,280.847992,43382.421596,97113.48,44.671884
3,ScheduledDomestic,AirAsia India,2021,4,Air Asia21.xlsx,Monthly_Template,3195.0,6341.383333,3481.586,354878.0,...,64.019154,2487.1216,169.238,2656.3596,27963.79866,3332.789591,208.044115,31504.632366,69631.72,45.244656
4,ScheduledDomestic,AirAsia India,2021,5,Air Asia21.xlsx,Monthly_Template,809.0,1562.15,867.183,63532.0,...,44.380178,1344.1482,57.5075,1401.6557,4039.994933,1744.005877,114.999778,5899.000588,17343.66,34.012432


,Type,Airline,Year,Month,Source File,Source Sheet,Aircraft Number,Aircraft Hours,Aircraft Kilometres,Passenger Number,...,Passenger Load Factor,Freight,Mail,Total Cargo,Passenger Tonne Kilometer,Mail Tonne Kilometer,Freight Tonne Kilometer,Total Tonne Kilometer,Available Tonne Kilometer,Weight Load Factor
0,ScheduledDomestic,AirAsia India,2021,1,Air Asia21.xlsx,Monthly_Template,4495.0,8590.4,4621.139,531578.0,...,66.829529,3175.7262,230.1885,3405.9147,37494.472877,4067.437374,269.242257,41831.152508,92422.78,45.260652
1,ScheduledDomestic,AirAsia India,2021,2,Air Asia21.xlsx,Monthly_Template,4340.0,8239.033333,4417.0,520603.0,...,67.890946,3410.6377,211.8922,3622.5299,39555.584103,4264.010976,251.988624,44071.583703,88341.12,49.887961
2,ScheduledDomestic,AirAsia India,2021,3,Air Asia21.xlsx,Monthly_Template,4727.0,9023.3,4855.674,541512.0,...,65.131994,3662.2624,211.7535,3874.0159,38524.441947,4577.131657,280.847992,43382.421596,97113.48,44.671884
3,ScheduledDomestic,AirAsia India,2021,4,Air Asia21.xlsx,Monthly_Template,3195.0,6341.383333,3481.586,354878.0,...,64.019154,2487.1216,169.238,2656.3596,27963.79866,3332.789591,208.044115,31504.632366,69631.72,45.244656
4,ScheduledDomestic,AirAsia India,2021,5,Air Asia21.xlsx,Monthly_Template,809.0,1562.15,867.183,63532.0,...,44.380178,1344.1482,57.5075,1401.6557,4039.994933,1744.005877,114.999778,5899.000588,17343.66,34.012432


In [12]:
KEYS=["Type","Airline","Year","Month"]
dupes=dgca[dgca.duplicated(KEYS,keep=False)].sort_values(KEYS)
print("Duplicate-key rows:",len(dupes))
if not dupes.empty:
    display(dupes[KEYS+["Source File","Source Sheet"]+METRICS].head(30))
    dupes.to_csv(INTERIM/f"duplicate_parse_records_{TODAY}.csv",index=False)
else:
    print("No duplicate records found.")

Duplicate-key rows: 0
No duplicate records found.


## 7. Cross-check against the GitHub table
This is the validation for the parser and the cross-source check for the cleaning section. Rows are matched on Type, Airline, Year, Month and the passenger counts compared.

In [13]:
import numpy as np
gh=pd.read_csv(RAW/"github"/gh_files["carrier"][1])
gh=gh[gh["Year"]>=START_YEAR].copy()
for d in [dgca,gh]:
    d["Type"]=d["Type"].astype("string").str.strip()
    d["Airline"]=d["Airline"].astype("string").str.strip()
    d["Year"]=pd.to_numeric(d["Year"],errors="coerce")
    d["Month"]=pd.to_numeric(d["Month"],errors="coerce")
    d["Passenger Number"]=pd.to_numeric(d["Passenger Number"].astype("string").str.replace(",","",regex=False),errors="coerce")
    d.dropna(subset=KEYS,inplace=True)
    d["Year"]=d["Year"].astype(int)
    d["Month"]=d["Month"].astype(int)

dgca_dupes=dgca[dgca.duplicated(KEYS,keep=False)]
gh_dupes=gh[gh.duplicated(KEYS,keep=False)]
print("DGCA duplicate keys:",len(dgca_dupes),"| GitHub duplicate keys:",len(gh_dupes))
if not dgca_dupes.empty or not gh_dupes.empty:
    raise ValueError("Resolve duplicate keys before performing the cross-source merge.")

m=dgca.merge(gh,on=KEYS,how="outer",suffixes=("_dgca","_gh"),indicator=True,validate="one_to_one")
print("\nRecord match status:\n",m["_merge"].value_counts().to_string())
both=m[m["_merge"]=="both"].copy()
a=both["Passenger Number_dgca"].to_numpy(dtype=float)
b=both["Passenger Number_gh"].to_numpy(dtype=float)
both["passenger_count_equal"]=np.isclose(a,b,rtol=0,atol=1e-6,equal_nan=True)
print(f"\nPassenger counts equal: {both.passenger_count_equal.sum()}/{len(both)} ({both.passenger_count_equal.mean():.1%})" if len(both) else "\nNo matched rows to compare.")
print("\nOnly in DGCA parse:\n",m[m["_merge"]=="left_only"].groupby(["Airline","Year"]).size().head(15).to_string())
print("\nOnly in GitHub:\n",m[m["_merge"]=="right_only"].groupby(["Airline","Year"]).size().head(15).to_string())
m.to_csv(INTERIM/f"cross_source_comparison_{TODAY}.csv",index=False)
display(both.loc[~both.passenger_count_equal,KEYS+["Passenger Number_dgca","Passenger Number_gh"]].head(20))

DGCA duplicate keys: 0 | GitHub duplicate keys: 0

Record match status:
 _merge
both          2966
left_only     2878
right_only     533

Passenger counts equal: 2966/2966 (100.0%)

Only in DGCA parse:
 Airline       Year
AIX Connect   2024    31
Air Carnival  2017     8
Air Costa     2017    10
Air Deccan    2017    11
              2020    47
              2021    48
Air Heritage  2019    42
              2020    45
              2021    48
Air India     2019    12
              2020    13
              2021    12
              2022    17
              2023    18
              2024    10

Only in GitHub:
 Airline             Year
Blue Dart Aviation  2017    12
                    2018    12
                    2019    16
                    2020    31
                    2021    34
                    2022    35
                    2023    24
                    2024    29
                    2025    24
                    2026    14
QuikJet Cargo       2017    24
                   

C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\457240493.py:10: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  d.dropna(subset=KEYS,inplace=True)
C:\Users\subhranshu das\AppData\Local\Temp\ipykernel_20652\457240493.py:10: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  d.dropna(subset=KEYS,inplace=True)


,Type,Airline,Year,Month,Passenger Number_dgca,Passenger Number_gh


## 8. Save the interim table
Cleaning (rubric section 3) is a separate step that reads this file, so the raw and interim stages stay reproducible.

In [14]:
dgca=dgca.sort_values(KEYS+["Source File","Source Sheet"]).reset_index(drop=True)
csv_path=INTERIM/f"dgca_domestic_airline_monthly_{TODAY}.csv"
parquet_path=INTERIM/f"dgca_domestic_airline_monthly_{TODAY}.parquet"
log_path=INTERIM/f"parse_log_{TODAY}.csv"
dgca.to_csv(csv_path,index=False)
dgca.to_parquet(parquet_path,index=False)
parse_log.to_csv(log_path,index=False)
check=pd.read_parquet(parquet_path)
pd.testing.assert_frame_equal(dgca,check,check_dtype=False)
print("CSV saved:",csv_path)
print("Parquet saved and read-back validated:",parquet_path)
print("Parse log saved:",log_path)
print("Final shape:",dgca.shape)
print("Unique airlines:",dgca.Airline.nunique())
print("Years:",dgca.Year.min(),"-",dgca.Year.max())
print("Traffic types:",dgca.Type.unique().tolist())

CSV saved: C:\Users\subhranshu das\OneDrive\Desktop\DGCA_aviation_analysis\data\interim\dgca_domestic_airline_monthly_2026-11-01.csv
Parquet saved and read-back validated: C:\Users\subhranshu das\OneDrive\Desktop\DGCA_aviation_analysis\data\interim\dgca_domestic_airline_monthly_2026-11-01.parquet
Parse log saved: C:\Users\subhranshu das\OneDrive\Desktop\DGCA_aviation_analysis\data\interim\parse_log_2026-11-01.csv
Final shape: (5844, 22)
Unique airlines: 27
Years: 2017 - 2026
Traffic types: ['NonScheduledDomestic', 'NonScheduledInternational', 'ScheduledDomestic', 'ScheduledInternational']


In [15]:
import json
metadata={"collection_date":TODAY,"start_year":START_YEAR,"include_citypair":INCLUDE_CITYPAIR,"excluded_names_pattern":EXCLUDE,"github_repo":"Vonter/india-aviation-traffic","github_commit":sha,"files_in_manifest":len(manifest),"downloaded":int((manifest.status=="downloaded").sum()),"cached":int((manifest.status=="cached").sum()),"failed":int((~manifest.status.isin(["downloaded","cached"])).sum()),"parsed_rows":len(dgca),"parsed_files":int((parse_log.status=="parsed").sum()),"files_with_parse_errors":int((parse_log.status=="error").sum()),"parquet_readback_validated":True}
with open(INTERIM/f"collection_metadata_{TODAY}.json","w") as f:
    json.dump(metadata,f,indent=2)
print(json.dumps(metadata,indent=2))

{
  "collection_date": "2026-11-01",
  "start_year": 2017,
  "include_citypair": false,
  "excluded_names_pattern": "totalint|quikjet|bluedart",
  "github_repo": "Vonter/india-aviation-traffic",
  "github_commit": "e197170f131e43169620e4f8aa520350d4cbcba9",
  "files_in_manifest": 150,
  "downloaded": 142,
  "cached": 0,
  "failed": 8,
  "parsed_rows": 5844,
  "parsed_files": 142,
  "files_with_parse_errors": 0,
  "parquet_readback_validated": true
}


## 9. City-pair passengers (for the route map)
The dashboard's route map needs passengers between pairs of cities. Vonter's table has them for most months. For any month it lacks, this section downloads DGCA's own city-pair file (Excel, or the PDF table when no Excel file is served), reads it, and saves one combined table to `data/interim/`. The dashboard reads only that table.

In [ ]:
# City-pair table for the route map: Vonter's table first, then DGCA's own files for any month it lacks
CITYPAIR_DIR = RAW / "dgca_citypair"; CITYPAIR_DIR.mkdir(parents=True, exist_ok=True)
MONTHLY_PATH = "/InventoryList/dataReports/aviationDataStatistics/airTransport/domestic/monthly/"
MONTH_NAMES = ["JANUARY", "FEBRUARY", "MARCH", "APRIL", "MAY", "JUNE", "JULY", "AUGUST", "SEPTEMBER", "OCTOBER", "NOVEMBER", "DECEMBER"]

# 1. Vonter's aggregated city-pair CSV (same commit as the airline tables; reuse a copy already on disk)
vonter_files = sorted((RAW / "github").glob("vonter_domestic_city_*.csv"))
if vonter_files:
    vonter_path = vonter_files[-1]; print("using the Vonter city-pair file already on disk:", vonter_path.name)
else:
    vonter_path = RAW / "github" / f"vonter_domestic_city_{TODAY}.csv"
    ref = sha if sha != "unknown" else "main"
    vonter_path.write_bytes(get(f"https://raw.githubusercontent.com/Vonter/india-aviation-traffic/{ref}/aggregated/domestic/city.csv").content)
    print("saved", vonter_path.name)
cols = ["Year", "Month", "City1", "City2", "PaxToCity2", "PaxFromCity2"]
vonter = pd.read_csv(vonter_path, usecols=cols).assign(Source="vonter")

# 2. months inside Vonter's range that it does not have
have = set(zip(vonter.Year, vonter.Month))
first = pd.Timestamp(year=START_YEAR, month=1, day=1); last = pd.Timestamp(year=int(vonter.Year.max()), month=int(vonter[vonter.Year == vonter.Year.max()].Month.max()), day=1)
missing = [d for d in pd.date_range(first, last, freq="MS") if (d.year, d.month) not in have]
print("months missing from the Vonter table:", [f"{d:%b %Y}" for d in missing])

# 3. fetch each one from DGCA's storage. The portal's link capitalisation is not always the one the server uses, so try the variants.
def fetch_citypair(d):
    cached = sorted(CITYPAIR_DIR.glob(f"dgca_citypair_{d.year}_{d.month:02d}_*"))
    if cached: return cached[-1]
    name = MONTH_NAMES[d.month - 1]
    for ext in (".xlsx", ".pdf"):
        for label in (name, name.title()):
            url = S3_BASE + quote(MONTHLY_PATH + f"DOM CITYPAIR DATA, {label} {d.year}{ext}", safe="/,")
            r = requests.get(url, headers=HEADERS, timeout=(15, 90))
            ok = r.status_code == 200 and (is_excel(r.content) if ext == ".xlsx" else r.content[:4] == b"%PDF")
            print(f"  {d:%b %Y} {label}{ext}: HTTP {r.status_code}" + ("  -> saved" if ok else ""))
            if ok:
                p = CITYPAIR_DIR / f"dgca_citypair_{d.year}_{d.month:02d}_{TODAY}{ext}"; p.write_bytes(r.content); return p
            time.sleep(0.5)
    return None

# 4. read a DGCA city-pair workbook or PDF table (the two have the same columns: S.No., City 1, City 2, passengers to / from City 2, ...)
def read_dgca_citypair(path, d):
    num = lambda v: pd.to_numeric(str(v).replace(",", "").strip(), errors="coerce")       # "-" means none
    if path.suffix == ".pdf":
        import pdfplumber
        with pdfplumber.open(path) as pdf: rows = [r for pg in pdf.pages for t in pg.extract_tables() for r in t]
    else:
        rows = pd.read_excel(path, header=None).values.tolist()
    rows = [r for r in rows if str(r[0]).strip().isdigit() and len(r) >= 5 and isinstance(r[1], str)]
    return pd.DataFrame({"Year": d.year, "Month": d.month, "City1": [r[1] for r in rows], "City2": [r[2] for r in rows],
                         "PaxToCity2": [num(r[3]) for r in rows], "PaxFromCity2": [num(r[4]) for r in rows], "Source": path.name})
extra = []
for d in missing:
    p = fetch_citypair(d)
    if p is None: print(f"  {d:%b %Y}: no file could be downloaded"); continue
    e = read_dgca_citypair(p, d); extra.append(e)
    print(f"  {d:%b %Y}: {len(e)} city pairs, {e.PaxToCity2.fillna(0).sum() + e.PaxFromCity2.fillna(0).sum():,.0f} passengers (from {p.name})")

In [ ]:
# Save the combined city-pair table, and check it against the airline-wise totals
citypair = pd.concat([vonter] + extra, ignore_index=True).sort_values(["Year", "Month", "City1", "City2"]).reset_index(drop=True)
citypair["Date"] = pd.to_datetime(dict(year=citypair.Year, month=citypair.Month, day=1))
cp_path = INTERIM / f"dgca_domestic_citypair_monthly_{TODAY}.csv"
citypair.drop(columns="Date").to_csv(cp_path, index=False)
print("saved", cp_path.name, "| rows:", len(citypair), "| months:", citypair.Date.nunique(), "| from DGCA files:", sorted(set(citypair.Source) - {"vonter"}))

airline_csv = sorted(INTERIM.glob("dgca_domestic_airline_monthly_*.csv"))[-1]
al = pd.read_csv(airline_csv); al = al[(al.Type == "ScheduledDomestic") & ~al.Airline.str.lower().str.contains("total")]
al_tot = al.assign(Date=pd.to_datetime(dict(year=al.Year, month=al.Month, day=1))).groupby("Date")["Passenger Number"].sum()
cp_tot = (citypair.PaxToCity2.fillna(0) + citypair.PaxFromCity2.fillna(0)).groupby(citypair.Date).sum()
ratio = (cp_tot / al_tot).dropna().loc["2019-01-01":]
print(f"city-pair total / airline total, 2019 onward: min {ratio.min():.3f}, max {ratio.max():.3f}  (should be about 1)")
print("months outside 0.98-1.02:", [f"{d:%b %Y}" for d in ratio.index[(ratio < .98) | (ratio > 1.02)]])